# Minimal LoRA XLM-R Classification + MAP Calibration

One-pass LoRA XLM-R baseline using a 5-class classifier instead of scalar regression. Labels are converted from class index `i` into the one-hot vector `e_i`, and training minimizes cross entropy against those one-hot targets.

At inference time we post-process posterior predictions with class-prior-constrained decoding. This keeps the probabilistic classifier but imposes the balanced class prior observed in the training data.

In [ ]:
# Run once on the cluster if needed.
# %pip install -q -U "transformers>=4.40" datasets accelerate "peft>=0.10" scikit-learn

In [ ]:
from pathlib import Path
import inspect
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, Sequence, Value
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from transformers import AutoConfig, AutoTokenizer, Trainer, TrainingArguments, XLMRobertaModel, XLMRobertaPreTrainedModel, set_seed

ROOT = Path.cwd()
if not (ROOT / "experiments").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

SEED = 42
MODEL_ID = "xlm-roberta-base"
TRAIN_CSV = ROOT / "data" / "train_lang.csv"
TEST_CSV = ROOT / "data" / "test.csv"
OUTPUT_DIR = ROOT / "outputs" / "minimal_lora_classification_map_xlmr"

# Use e.g. 20000 for a smoke test; None for full data.
SAMPLE_N = None
VAL_SIZE = 0.10
MAX_LENGTH = 128
N_CLASSES = 5

EPOCHS = 1
BATCH_SIZE = 64
EVAL_BATCH_SIZE = 1024
LR = 1.5e-4
FP16 = torch.cuda.is_available()

# Larger values enforce the balanced prediction prior more strongly.
MAP_GAMMA = 0.02

os.environ.setdefault("WANDB_DISABLED", "true")
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Data and tokenization

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df["sentence"] = df["sentence"].fillna("")
df["lang"] = df.get("lang", "unk")

if SAMPLE_N is not None and SAMPLE_N < len(df):
    df, _ = train_test_split(df, train_size=SAMPLE_N, random_state=SEED, stratify=df["label"])

train_df, val_df = train_test_split(df, test_size=VAL_SIZE, random_state=SEED, stratify=df["label"])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("train", train_df.shape, "val", val_df.shape)
print(train_df["label"].value_counts().sort_index().to_dict())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def one_hot(label, n_classes=N_CLASSES):
    vec = [0.0] * n_classes
    vec[int(label)] = 1.0
    return vec


def tokenize_hard(batch):
    out = tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    out["labels"] = [one_hot(x) for x in batch["label"]]
    langs = batch["lang"] if "lang" in batch else ["unk"] * len(batch["sentence"])
    out["lang"] = [0 if x == "eng_Latn" else 1 for x in langs]
    return out


def to_hf_dataset(frame):
    ds = Dataset.from_pandas(frame, preserve_index=False)
    ds = ds.map(tokenize_hard, batched=True, remove_columns=ds.column_names)
    ds = ds.cast_column("labels", Sequence(Value("float32"), length=N_CLASSES))
    ds = ds.cast_column("lang", Value("int64"))
    ds.set_format("torch")
    return ds


train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)

## Model and trainer

In [ ]:
class XLMRClassificationModel(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = XLMRobertaModel(config)
        hidden = config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden // 2, hidden // 4),
            nn.GELU(),
            nn.Linear(hidden // 4, config.num_labels),
        )
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            log_probs = F.log_softmax(logits, dim=-1)
            loss = -(labels.float() * log_probs).sum(dim=-1).mean()
        return {"loss": loss, "logits": logits}


def make_model():
    config = AutoConfig.from_pretrained(MODEL_ID, num_labels=N_CLASSES)
    model = XLMRClassificationModel.from_pretrained(MODEL_ID, config=config)
    lora_config = LoraConfig(
        r=128,
        lora_alpha=64,
        target_modules=["query", "key", "value", "intermediate.dense", "output.dense"],
        modules_to_save=["classifier"],
        lora_dropout=0.01,
        task_type="SEQ_CLS",
    )
    model = get_peft_model(model, lora_config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
    return model


def softmax_np(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def expected_mae_risk(probs):
    classes = np.arange(probs.shape[1])
    return np.stack([np.sum(probs * np.abs(cls - classes), axis=1) for cls in classes], axis=1)


def bayes_mae_decode(probs):
    return expected_mae_risk(probs).argmin(axis=1).astype(int)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax_np(logits)
    true = np.asarray(labels).argmax(axis=1)
    map_preds = probs.argmax(axis=1)
    mae_preds = bayes_mae_decode(probs)
    expected_score = probs @ np.arange(N_CLASSES)
    return {
        "accuracy": float(accuracy_score(true, map_preds)),
        "map_mae": float(mean_absolute_error(true, map_preds)),
        "bayes_mae": float(mean_absolute_error(true, mae_preds)),
        "expected_score_mae": float(mean_absolute_error(true, expected_score)),
    }


def make_training_args(run_name):
    kwargs = dict(
        output_dir=str(OUTPUT_DIR / run_name / "checkpoints"),
        overwrite_output_dir=True,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        num_train_epochs=EPOCHS,
        evaluation_strategy="steps",
        eval_steps=500,
        logging_steps=100,
        save_strategy="epoch",
        save_total_limit=1,
        fp16=FP16,
        report_to=[],
        remove_unused_columns=False,
        seed=SEED,
    )
    params = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    return TrainingArguments(**kwargs)

## Train for 1 epoch

In [ ]:
model = make_model()
trainer = Trainer(
    model=model,
    args=make_training_args("classification_1epoch"),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
trainer.train()
# metrics = trainer.evaluate()
# metrics

## MAP calibration with a balanced prediction prior

We post-process calibrated posterior predictions using class-prior-constrained decoding. Since the observed training labels are balanced, we impose a balanced prediction prior on the test set. The constrained decoder changes only low-regret examples, i.e. examples whose posterior probabilities or expected MAE risks make multiple adjacent labels plausible.

Mathematically, the decoder solves the regularized assignment objective

$$
\hat{y} = \arg\min_{\hat{y}_1,\ldots,\hat{y}_N}
\sum_{i=1}^{N}\sum_{k=0}^{4} p_i(k)\,|\hat{y}_i-k|
+ \gamma\sum_{k=0}^{4}\left(\hat{n}_k - \frac{N}{5}\right)^2,
$$

where $p_i(k)$ is the calibrated posterior probability of class $k$, $\hat{n}_k$ is the number of decoded predictions assigned to class $k$, and $\gamma$ controls the strength of the balanced-prior penalty.

In [ ]:
def objective_value(risk, preds, gamma):
    counts = np.bincount(preds, minlength=N_CLASSES).astype(float)
    target = len(preds) / N_CLASSES
    return float(risk[np.arange(len(preds)), preds].sum() + gamma * np.square(counts - target).sum())


def class_prior_constrained_decode(probs, gamma=MAP_GAMMA, max_iter=10000, tol=1e-12):
    """Decode posteriors with expected-MAE risk and a quadratic balanced-prior penalty.

    Starts from the Bayes estimator for MAE, then repeatedly applies the single-label
    move with the largest objective decrease. The examples moved first are exactly the
    low-regret examples under the posterior risk matrix.
    """
    probs = np.asarray(probs, dtype=np.float64)
    risk = expected_mae_risk(probs)
    preds = risk.argmin(axis=1).astype(int)
    counts = np.bincount(preds, minlength=N_CLASSES).astype(int)
    target = len(preds) / N_CLASSES

    for _ in range(max_iter):
        best_delta = 0.0
        best_i = None
        best_cls = None
        current_risk = risk[np.arange(len(preds)), preds]

        for cls in range(N_CLASSES):
            from_cls = preds
            same = from_cls == cls
            if same.all():
                continue
            old_penalty = np.square(counts[from_cls] - target) + np.square(counts[cls] - target)
            new_penalty = np.square(counts[from_cls] - 1 - target) + np.square(counts[cls] + 1 - target)
            penalty_delta = gamma * (new_penalty - old_penalty)
            delta = risk[:, cls] - current_risk + penalty_delta
            delta[same] = np.inf
            i = int(np.argmin(delta))
            if delta[i] < best_delta - tol:
                best_delta = float(delta[i])
                best_i = i
                best_cls = cls

        if best_i is None:
            break

        old_cls = int(preds[best_i])
        preds[best_i] = int(best_cls)
        counts[old_cls] -= 1
        counts[best_cls] += 1

    return preds, counts, objective_value(risk, preds, gamma), risk


def summarize_decoding(labels, probs, gamma=MAP_GAMMA):
    labels = np.asarray(labels, dtype=int)
    map_preds = probs.argmax(axis=1).astype(int)
    bayes_preds = bayes_mae_decode(probs)
    constrained_preds, counts, obj, risk = class_prior_constrained_decode(probs, gamma=gamma)

    summary = pd.DataFrame(
        [
            {"decoder": "posterior_map", "mae": mean_absolute_error(labels, map_preds), "counts": np.bincount(map_preds, minlength=N_CLASSES).tolist()},
            {"decoder": "bayes_expected_mae", "mae": mean_absolute_error(labels, bayes_preds), "counts": np.bincount(bayes_preds, minlength=N_CLASSES).tolist()},
            {"decoder": f"balanced_prior_gamma_{gamma:g}", "mae": mean_absolute_error(labels, constrained_preds), "counts": counts.tolist()},
        ]
    )
    changed = np.flatnonzero(constrained_preds != bayes_preds)
    regrets = risk[changed, constrained_preds[changed]] - risk[changed, bayes_preds[changed]] if len(changed) else np.array([])
    print(f"changed from Bayes-MAE decoder: {len(changed)} / {len(labels)}")
    if len(regrets):
        print("move regret quantiles:", dict(zip([0, 0.5, 0.9, 1.0], np.quantile(regrets, [0, 0.5, 0.9, 1.0]).round(6))))
    print("objective:", round(obj, 4))
    return summary, constrained_preds, risk

In [ ]:
val_logits = trainer.predict(val_ds).predictions
val_probs = softmax_np(val_logits)
val_labels = val_df["label"].to_numpy(dtype=int)

summary, val_constrained_preds, val_risk = summarize_decoding(val_labels, val_probs, gamma=MAP_GAMMA)
display(summary)

pd.crosstab(
    pd.Series(val_labels, name="label"),
    pd.Series(val_constrained_preds, name="balanced_prior_pred"),
    margins=True,
)

In [ ]:
final_dir = OUTPUT_DIR / "final_model"
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
calibration_path = OUTPUT_DIR / "map_calibration_config.json"
calibration_path.write_text(
    pd.Series({"gamma": MAP_GAMMA, "n_classes": N_CLASSES, "decoder": "class_prior_constrained_expected_mae"}).to_json(indent=2),
    encoding="utf-8",
)
print("model:", final_dir)
print("calibration:", calibration_path)

## Optional submission with MAP-calibrated decoding

In [ ]:
if TEST_CSV.exists():
    test_df = pd.read_csv(TEST_CSV)
    test_df["sentence"] = test_df["sentence"].fillna("")
    test_ds_raw = Dataset.from_pandas(test_df, preserve_index=False)

    def tokenize_test(batch):
        return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

    test_ds = test_ds_raw.map(tokenize_test, batched=True, remove_columns=test_ds_raw.column_names)
    test_ds.set_format("torch")
    test_logits = trainer.predict(test_ds).predictions
    test_probs = softmax_np(test_logits)
    test_preds, test_counts, test_obj, test_risk = class_prior_constrained_decode(test_probs, gamma=MAP_GAMMA)

    submission = pd.DataFrame({"id": test_df["id"], "label": test_preds.astype(int)})
    submission_path = OUTPUT_DIR / "submission_classification_map.csv"
    submission.to_csv(submission_path, index=False)
    print("counts:", test_counts.tolist())
    print("objective:", round(test_obj, 4))
    print(submission_path)
    display(submission.head())